https://jalammar.github.io/illustrated-bert/

### Text Classification

In [ ]:
from google.colab import userdata
hf_token = userdata.get('HF_Access_Token')

In [ ]:
# Manually access hugging face
!pip install huggingface_hub

In [ ]:
# Login to Hugging Face
import huggingface_hub
huggingface_hub.login()

In [ ]:
try:
  import datasets, evaluate, accelerate
except:
  !pip install -U datasets evaluate accelerate
  import datasets, evaluate, accelerate

import numpy as np
import pandas as pd
import transformers
import tensorflow as tf

transformers.__version__, tf.__version__, datasets.__version__


('4.57.3', '2.19.0', '4.4.2')

Import Dataset

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [ ]:
# To inspect features
dataset.column_names

{'train': ['text', 'label'],
 'test': ['text', 'label'],
 'unsupervised': ['text', 'label']}

In [ ]:
# Train
dataset['train']

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [ ]:
dataset['train'][10]

{'text': 'It was great to see some of my favorite stars of 30 years ago including John Ritter, Ben Gazarra and Audrey Hepburn. They looked quite wonderful. But that was it. They were not given any characters or good lines to work with. I neither understood or cared what the characters were doing.<br /><br />Some of the smaller female roles were fine, Patty Henson and Colleen Camp were quite competent and confident in their small sidekick parts. They showed some talent and it is sad they didn\'t go on to star in more and better films. Sadly, I didn\'t think Dorothy Stratten got a chance to act in this her only important film role.<br /><br />The film appears to have some fans, and I was very open-minded when I started watching it. I am a big Peter Bogdanovich fan and I enjoyed his last movie, "Cat\'s Meow" and all his early ones from "Targets" to "Nickleodeon". So, it really surprised me that I was barely able to keep awake watching this one.<br /><br />It is ironic that this movie is a

In [ ]:
dataset['test'][0]

{'text': 'I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as 

In [ ]:
rand_arr = np.random.randint(1, 1000, (5,))
rand_arr

array([610, 449, 486, 379, 168])

In [ ]:
dataset['train'][rand_arr]

{'text': ["I found this film to be quite an oddity. From the very get go I found it extremely hard to like this movie, and now after a little thinking about it I can pretty much pinpoint the reason why. Jean-Marc Barr, although I love him to bits (I think Zentropa is one of the best movies ever made) is quite miscast here, and although I can't figure for the life of me who would be better, I am sure someone could have taken his place quite easily and make this film work. Everything else is fine, except for the stabs at weak comedy (A Meet The Parents Joke is not really needed, filmmakers!) and I really like Richard E. Grant as the British Major. It just suffers from one thing.. Jean-Marc.",
  'This was disappointing. It started well enough but as it went on and lost every opportunity to soar, it fell flat. Maria Schrader\'s acting is dreadful, never seeming to mean what she says, or even knowing what she says until she says it. She showed no genuine emotion at all, not for her beloved 

In [ ]:
dataset['train'][rand_arr]['text']

["I found this film to be quite an oddity. From the very get go I found it extremely hard to like this movie, and now after a little thinking about it I can pretty much pinpoint the reason why. Jean-Marc Barr, although I love him to bits (I think Zentropa is one of the best movies ever made) is quite miscast here, and although I can't figure for the life of me who would be better, I am sure someone could have taken his place quite easily and make this film work. Everything else is fine, except for the stabs at weak comedy (A Meet The Parents Joke is not really needed, filmmakers!) and I really like Richard E. Grant as the British Major. It just suffers from one thing.. Jean-Marc.",
 'This was disappointing. It started well enough but as it went on and lost every opportunity to soar, it fell flat. Maria Schrader\'s acting is dreadful, never seeming to mean what she says, or even knowing what she says until she says it. She showed no genuine emotion at all, not for her beloved goy, or he

In [ ]:
dataset['train'][rand_arr]['label']

[0, 0, 0, 0, 0]

In [ ]:
len(dataset['train'])

25000

Tokenize the text and labels

In [ ]:
# Create mappings: Convert labels to ids/numbers
idx_to_label = {}
for idx, label in enumerate(dataset['train'].unique('label')):
  idx_to_label[idx] = label
idx_to_label

{0: 0, 1: 1}

Segregate into Train and Validation split

In [ ]:
# Use huggingface train_test_split
train_valid = dataset['train'].train_test_split(test_size=0.2, seed = 42, stratify_by_column = 'label')
train_valid

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 20000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
})

In [ ]:
# Convert train and test to train and validation
from datasets import DatasetDict

In [ ]:
splits = DatasetDict({'train':train_valid['train'],
                      'valid':train_valid['test'],
                      'test': dataset['test']})
splits

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 20000
    })
    valid: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})

Tokenization

https://platform.openai.com/tokenizer

In [ ]:
from transformers import AutoTokenizer

In [ ]:
# Models are often paired with tokenizers
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path= 'distilbert/distilbert-base-uncased', use_fast = True)
tokenizer

DistilBertTokenizerFast(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [ ]:
# Test the tokenizer
tokenizer('Hello, how are you?')

{'input_ids': [101, 7592, 1010, 2129, 2024, 2017, 1029, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokenizer('hello, what are you doing?')

{'input_ids': [101, 7592, 1010, 2054, 2024, 2017, 2725, 1029, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokenizer('my name is sreeram')

{'input_ids': [101, 2026, 2171, 2003, 5034, 11510, 3286, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokenizer('my name is sree')

{'input_ids': [101, 2026, 2171, 2003, 5034, 4402, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokenizer('my name is sree ram')

{'input_ids': [101, 2026, 2171, 2003, 5034, 4402, 8223, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokenizer.vocab['sreeram']

KeyError: 'sreeram'

In [ ]:
tokenizer.vocab['tom']

3419

In [ ]:
tokenizer.vocab['sri']

5185

In [ ]:
tokenizer.vocab['sree']

KeyError: 'sree'

In [ ]:
tokenizer.vocab['ram']

8223

In [ ]:
tokenizer('my name is sree ra m')

{'input_ids': [101, 2026, 2171, 2003, 5034, 4402, 10958, 1049, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokenizer.vocab['hugging face']

KeyError: 'hugging face'

In [ ]:
tokenizer.vocab['huggingface']

KeyError: 'huggingface'

In [ ]:
tokenizer('hugging face')

{'input_ids': [101, 17662, 2227, 102], 'attention_mask': [1, 1, 1, 1]}

In [ ]:
tokenizer.convert_ids_to_tokens(tokenizer('hugging face')['input_ids'])

['[CLS]', 'hugging', 'face', '[SEP]']

In [ ]:
tokenizer.convert_ids_to_tokens(tokenizer('sreeram')['input_ids'])

['[CLS]', 'sr', '##eer', '##am', '[SEP]']

In [ ]:
tokenizer.convert_ids_to_tokens(tokenizer('✌️')['input_ids'])

['[CLS]', '[UNK]', '[SEP]']

In [ ]:
sorted(tokenizer.vocab.items())[:5]

[('!', 999), ('"', 1000), ('#', 1001), ('##!', 29612), ('##"', 29613)]

In [ ]:
print(tokenizer.vocab)

{'sewing': 22746, 'couple': 3232, 'patsy': 25382, 'kapoor': 17129, '8': 1022, 'georg': 12062, '[unused785]': 790, 'サ': 1705, 'restore': 9239, '##nally': 26827, 'measurement': 10903, '##ー': 30265, 'positions': 4460, 'salute': 17664, 'psychic': 12663, '##jou': 23099, '##hedron': 26440, 'த': 1385, 'electrical': 5992, 'bauer': 17838, '##9': 2683, 'ད': 1428, 'adventurer': 29506, 'crowds': 12783, 'defended': 8047, 'rhapsody': 29395, 'bounds': 19202, 'structurally': 29060, 'barony': 19365, 'stores': 5324, '1945': 3386, '##inking': 29377, 'macy': 20914, 'citation': 11091, 'killer': 6359, 'deposition': 19806, 'peck': 18082, 'immersed': 26275, 'enthusiastically': 24935, 'rubbish': 29132, 'pitchers': 23232, 'celebrations': 12035, 'deutschland': 28668, 'streaks': 21295, 'tame': 24763, 'aba': 19557, 'agreed': 3530, 'father': 2269, 'claude': 8149, 'ur': 24471, '##松': 30406, '##zon': 11597, 'securely': 28999, 'reviewing': 15252, '##bee': 11306, 'narrative': 7984, 'leasing': 26707, 'sonora': 26647, '#

In [ ]:
print(dir(tokenizer))

['SPECIAL_TOKENS_ATTRIBUTES', '__annotations__', '__bool__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_add_tokens', '_auto_class', '_batch_encode_plus', '_call_one', '_convert_encoding', '_convert_id_to_token', '_convert_token_to_id_with_added_voc', '_create_repo', '_decode', '_decode_use_source_tokenizer', '_encode_plus', '_eventual_warn_about_too_long_sequence', '_eventually_correct_t5_max_length', '_from_pretrained', '_get_files_timestamps', '_get_padding_truncation_strategies', '_in_target_context_manager', '_pad', '_pad_token_type_id', '_patch_mistral_regex', '_processor_class', '_save_pretrained', '_set_model_specific_special_to

In [ ]:
tokenizer.vocab_size

30522

In [ ]:
tokenizer.model_max_length

512

In [ ]:
def tokenize_text(data):
  return tokenizer(data['text'], padding = True, truncation= True)

In [ ]:
tokenize_text(splits['train'][1])

{'input_ids': [101, 2000, 2022, 4189, 2027, 2106, 2004, 2092, 2004, 2027, 2071, 2007, 1037, 5166, 1997, 2274, 29332, 1998, 2416, 11837, 3401, 1010, 2021, 1996, 7982, 2001, 2062, 18178, 2229, 2100, 2084, 1023, 20850, 2015, 1997, 7861, 26901, 1998, 1996, 1039, 5856, 2001, 1037, 2210, 2214, 6045, 2085, 1012, 2672, 2065, 2070, 1997, 1996, 5889, 2020, 2025, 2061, 6669, 9610, 19358, 2094, 2041, 1997, 9753, 2009, 2052, 2031, 2081, 1996, 2143, 1037, 2210, 2488, 2205, 1012, 1012, 2000, 2360, 2023, 2001, 9643, 2003, 2000, 2079, 2023, 2143, 1037, 28616, 1011, 2326, 1010, 2065, 2017, 2215, 2000, 2156, 2242, 2008, 2003, 6135, 4654, 8586, 16670, 1010, 2017, 10657, 4133, 1998, 5949, 1037, 3232, 1997, 2847, 1997, 2115, 2166, 3666, 1005, 5305, 2571, 1005, 1010, 2008, 2003, 17111, 2568, 15903, 15787, 9643, 1010, 2049, 2941, 2204, 1010, 1006, 2195, 2312, 14813, 21705, 2024, 4315, 8004, 13094, 2295, 1012, 2151, 2346, 2039, 1010, 1045, 5632, 2023, 2143, 1998, 2049, 10657, 2022, 4276, 1037, 2298, 2065, 2017

In [ ]:
# Tokenize the whole data
tokenized_dataset = dataset.map(function = tokenize_text,
                                 batched = True,
                                 batch_size = 128)
tokenized_dataset

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 50000
    })
})

In [ ]:
tokenized_dataset = splits.map(function = tokenize_text,
                                 batched = True,
                                 batch_size = 128)
tokenized_dataset

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 20000
    })
    valid: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
})

In [ ]:
print(tokenized_dataset['train'][0]['text'])

After reading tons of good reviews about this movie I decided to take it for a spin (I bought it on DVD, hence the "spin" pun...I'm a dork). The beginning was everything I hoped for, a perfect set-up (along with some quotes that I've heard on Various Wu-Tang albums) to what should have been a good movie. But the plot I heard was so great, was so predictable. Every time I saw a character (except for the Lizard) I guessed which Venom he was. Plus, the only cool character gets killed off in the middle of the movie. Ok, so the plot wasn't very good but at least there was some good kung-fu right? Wrong. The fights were very short and few and far between. Granted the different styles were all pretty cool but I wish the fights were longer. I kept hoping to see the Lizard run and do some crazy ish on the walls but it never happened. I was hoping to see the Centipede do some tight speedy ish but it never happened. I was hoping to see the Scorpion in the movie for more than 7 total minutes but i

In [ ]:
print(tokenized_dataset['train'][0]['label'])

1


In [ ]:
print(tokenized_dataset['train'][0]['input_ids'])

[101, 2044, 3752, 6197, 1997, 2204, 4391, 2055, 2023, 3185, 1045, 2787, 2000, 2202, 2009, 2005, 1037, 6714, 1006, 1045, 4149, 2009, 2006, 4966, 1010, 6516, 1996, 1000, 6714, 1000, 26136, 1012, 1012, 1012, 1045, 1005, 1049, 1037, 2079, 8024, 1007, 1012, 1996, 2927, 2001, 2673, 1045, 5113, 2005, 1010, 1037, 3819, 2275, 1011, 2039, 1006, 2247, 2007, 2070, 16614, 2008, 1045, 1005, 2310, 2657, 2006, 2536, 8814, 1011, 9745, 4042, 1007, 2000, 2054, 2323, 2031, 2042, 1037, 2204, 3185, 1012, 2021, 1996, 5436, 1045, 2657, 2001, 2061, 2307, 1010, 2001, 2061, 21425, 1012, 2296, 2051, 1045, 2387, 1037, 2839, 1006, 3272, 2005, 1996, 15450, 1007, 1045, 11445, 2029, 15779, 2002, 2001, 1012, 4606, 1010, 1996, 2069, 4658, 2839, 4152, 2730, 2125, 1999, 1996, 2690, 1997, 1996, 3185, 1012, 7929, 1010, 2061, 1996, 5436, 2347, 1005, 1056, 2200, 2204, 2021, 2012, 2560, 2045, 2001, 2070, 2204, 18577, 1011, 11865, 2157, 1029, 3308, 1012, 1996, 9590, 2020, 2200, 2460, 1998, 2261, 1998, 2521, 2090, 1012, 4379, 19

In [ ]:
print(tokenized_dataset['train'][0]['attention_mask'])

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

Evaluation metric

In [ ]:
accuracy_score = evaluate.load('accuracy')
accuracy_score

EvaluationModule(name: "accuracy", module_type: "metric", features: {'predictions': Value('int32'), 'references': Value('int32')}, usage: """
Args:
    predictions (`list` of `int`): Predicted labels.
    references (`list` of `int`): Ground truth labels.
    normalize (`boolean`): If set to False, returns the number of correctly classified samples. Otherwise, returns the fraction of correctly classified samples. Defaults to True.
    sample_weight (`list` of `float`): Sample weights Defaults to None.

Returns:
    accuracy (`float` or `int`): Accuracy score. Minimum possible value is 0. Maximum possible value is 1.0, or the number of examples input, if `normalize` is set to `True`.. A higher score means higher accuracy.

Examples:

    Example 1-A simple example
        >>> accuracy_metric = evaluate.load("accuracy")
        >>> results = accuracy_metric.compute(references=[0, 1, 2, 0, 1, 2], predictions=[0, 1, 1, 2, 1, 0])
        >>> print(results)
        {'accuracy': 0.5}

    Exa

In [ ]:
def compute_accuracy(pred_and_labels):
  preds, labels = pred_and_labels
  return accuracy_score.compute(predictions = preds, references = labels)

In [ ]:
preds_and_labels = [1,1,1,0,0,0], [1,0,1,0,1,0]
compute_accuracy(preds_and_labels)

{'accuracy': 0.6666666666666666}

Load a Model

In [ ]:
from transformers import AutoModelForSequenceClassification

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(pretrained_model_name_or_path= 'distilbert/distilbert-base-uncased')
model

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(pretrained_model_name_or_path= 'distilbert/distilbert-base-uncased',
                                                           num_labels = 2)
model

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
print(tokenized_dataset['train'][0]['input_ids'])

[101, 2044, 3752, 6197, 1997, 2204, 4391, 2055, 2023, 3185, 1045, 2787, 2000, 2202, 2009, 2005, 1037, 6714, 1006, 1045, 4149, 2009, 2006, 4966, 1010, 6516, 1996, 1000, 6714, 1000, 26136, 1012, 1012, 1012, 1045, 1005, 1049, 1037, 2079, 8024, 1007, 1012, 1996, 2927, 2001, 2673, 1045, 5113, 2005, 1010, 1037, 3819, 2275, 1011, 2039, 1006, 2247, 2007, 2070, 16614, 2008, 1045, 1005, 2310, 2657, 2006, 2536, 8814, 1011, 9745, 4042, 1007, 2000, 2054, 2323, 2031, 2042, 1037, 2204, 3185, 1012, 2021, 1996, 5436, 1045, 2657, 2001, 2061, 2307, 1010, 2001, 2061, 21425, 1012, 2296, 2051, 1045, 2387, 1037, 2839, 1006, 3272, 2005, 1996, 15450, 1007, 1045, 11445, 2029, 15779, 2002, 2001, 1012, 4606, 1010, 1996, 2069, 4658, 2839, 4152, 2730, 2125, 1999, 1996, 2690, 1997, 1996, 3185, 1012, 7929, 1010, 2061, 1996, 5436, 2347, 1005, 1056, 2200, 2204, 2021, 2012, 2560, 2045, 2001, 2070, 2204, 18577, 1011, 11865, 2157, 1029, 3308, 1012, 1996, 9590, 2020, 2200, 2460, 1998, 2261, 1998, 2521, 2090, 1012, 4379, 19

In [ ]:
print(len(tokenized_dataset['train'][0]['input_ids']))

512


In [ ]:
type(model.parameters())

generator

In [ ]:
param_count = 0
for layer in model.parameters():
  if layer.requires_grad:
    param_count += layer.numel()
param_count

66955010

In [ ]:
# Create path to save path
from pathlib import Path
dir_path = Path('/content')
model_name = 'saved_model'
save_path = Path(dir_path, model_name)
save_path

PosixPath('/content/saved_model')

Train the model

In [ ]:
from transformers import TrainingArguments

````
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="your-model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True,
)
````

In [ ]:
training_args = TrainingArguments(
    output_dir= dir_path,
    learning_rate = 0.00001,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=16,
    save_total_limit = 3,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end = True,
    use_cpu = False,
    )

In [ ]:
training_args